<a href="https://colab.research.google.com/github/melissa-04/melisayla-biyoinformatik/blob/main/notebooks/rna-seq/mutfak/00_veri_hazirlama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Veri hazırlama: küçültülmüş FASTQ'lar (üretici defteri)

**Bu defter okuyucular için değil, siteyi hazırlayan için.** GSE33979 (Klf1 knockout vs. yabani tip, fare fetal karaciğeri) veri setindeki 6 örneğin her birinden ilk **1 milyon okumayı** ENA'dan akış halinde çeker ve Google Drive'ına kaydeder. Dosyaların tamamı indirilmez; örnek başına 1–2 dakika sürer.

Çıktılar: `SRR38497x_1M.fastq.gz` × 6, `ornekler.csv` (örnek tablosu), `md5sum.txt` (doğrulama). Bu klasör olduğu gibi Zenodo'ya yüklenecek.

**Colab 101:** Her kutu bir "hücre". Sol üstündeki ▶ düğmesine basarak (ya da hücre seçiliyken Shift+Enter) çalıştırılır. Hücreleri **yukarıdan aşağıya sırayla** çalıştır. Sol tarafta `[*]` görünüyorsa hâlâ çalışıyor demektir, bekle. Oturum kapanırsa (uzun süre boş kalınca) baştan başla; Drive'a kaydedilenler kaybolmaz.

## 1. Drive'ı bağla
Çalıştırınca izin penceresi açılır; Google hesabını seçip izin ver. Kalıcı dosyalar bu klasöre yazılacak.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
HEDEF = '/content/drive/MyDrive/melisa-ile-biyoinformatik/rna-seq/veri_1M'
os.makedirs(HEDEF, exist_ok=True)
print('Kayıt klasörü:', HEDEF)

Mounted at /content/drive
Kayıt klasörü: /content/drive/MyDrive/melisa-ile-biyoinformatik/rna-seq/veri_1M


## 2. Örnek tablosu
GEO/SRA Run Selector'dan okuduğumuz bilgiler. Bu tablo rehberin "Veri" sayfasında da yayımlanacak.

In [ ]:
import pandas as pd

ornekler = pd.DataFrame({
    'run':      ['SRR384977','SRR384978','SRR384979','SRR384980','SRR384981','SRR384982'],
    'gsm':      ['GSM839870','GSM839871','GSM839872','GSM839873','GSM839874','GSM839875'],
    'genotip':  ['WT','WT','WT','KO','KO','KO'],
    'kutuphane':['WT library 1','WT library 2','WT library 3','KO library 1','KO library 2','KO library 3'],
})
ornekler.to_csv(f'{HEDEF}/ornekler.csv', index=False)
ornekler

,run,gsm,genotip,kutuphane
0,SRR384977,GSM839870,WT,WT library 1
1,SRR384978,GSM839871,WT,WT library 2
2,SRR384979,GSM839872,WT,WT library 3
3,SRR384980,GSM839873,KO,KO library 1
4,SRR384981,GSM839874,KO,KO library 2
5,SRR384982,GSM839875,KO,KO library 3


## 3. ENA'dan indirme bağlantılarını al
SRA'nın Avrupa aynası ENA, FASTQ dosyalarını doğrudan verir. Bağlantıları elle yazmak yerine ENA'nın API'sinden çekiyoruz; böylece adres değişirse bile defter çalışır.

In [ ]:
import io, requests

url = ('https://www.ebi.ac.uk/ena/portal/api/filereport'
       '?accession=SRP009464&result=read_run'
       '&fields=run_accession,fastq_ftp,fastq_bytes,read_count&format=tsv')
ena = pd.read_csv(io.StringIO(requests.get(url, timeout=60).text), sep='\t')
ena = ena.rename(columns={'run_accession':'run'})
tablo = ornekler.merge(ena, on='run')
tablo['okuma_sayisi_milyon'] = (tablo['read_count'] / 1e6).round(1)
tablo['boyut_GB'] = (tablo['fastq_bytes'].astype(float) / 1e9).round(2)
tablo[['run','genotip','okuma_sayisi_milyon','boyut_GB','fastq_ftp']]

,run,genotip,okuma_sayisi_milyon,boyut_GB,fastq_ftp
0,SRR384977,WT,20.0,0.69,ftp.sra.ebi.ac.uk/vol1/fastq/SRR384/SRR384977/...
1,SRR384978,WT,22.2,0.73,ftp.sra.ebi.ac.uk/vol1/fastq/SRR384/SRR384978/...
2,SRR384979,WT,45.0,1.55,ftp.sra.ebi.ac.uk/vol1/fastq/SRR384/SRR384979/...
3,SRR384980,KO,25.2,1.73,ftp.sra.ebi.ac.uk/vol1/fastq/SRR384/SRR384980/...
4,SRR384981,KO,26.3,0.87,ftp.sra.ebi.ac.uk/vol1/fastq/SRR384/SRR384981/...
5,SRR384982,KO,26.0,1.67,ftp.sra.ebi.ac.uk/vol1/fastq/SRR384/SRR384982/...


## 4. Her örnekten ilk 1 milyon okumayı çek
FASTQ'da her okuma 4 satırdır; 1 milyon okuma = 4 milyon satır. `curl` dosyayı akış halinde okur, `head` ilk 4 milyon satırı alınca akış kesilir. Not: bu **konumsal** bir alt küme (dosyanın başı), rastgele değil; kalite kontrol ve kantifikasyon adımlarını *göstermek* için yeterlidir. İstatistik için tam veri kullanılacak.

In [ ]:
N_OKUMA = 1_000_000
N_SATIR = N_OKUMA * 4

for _, r in tablo.iterrows():
    kaynak = 'https://' + r['fastq_ftp']
    hedef  = f"{HEDEF}/{r['run']}_1M.fastq.gz"
    if os.path.exists(hedef):
        print('zaten var, atlanıyor:', hedef); continue
    print(f"{r['run']} ({r['genotip']}) indiriliyor...")
    !curl -sL "{kaynak}" | zcat 2>/dev/null | head -n {N_SATIR} | gzip > "{hedef}"
    print('  bitti:', round(os.path.getsize(hedef)/1e6, 1), 'MB')

SRR384977 (WT) indiriliyor...
  bitti: 34.9 MB
SRR384978 (WT) indiriliyor...
  bitti: 32.7 MB
SRR384979 (WT) indiriliyor...
  bitti: 34.4 MB
SRR384980 (KO) indiriliyor...
  bitti: 69.1 MB
SRR384981 (KO) indiriliyor...
  bitti: 33.5 MB
SRR384982 (KO) indiriliyor...
  bitti: 65.6 MB


## 5. Doğrulama
Her dosyada tam 1.000.000 okuma olmalı. md5 listesi Zenodo'ya da yüklenecek; indiren kişi dosyanın bozulmadığını bununla kontrol eder.

In [ ]:
for _, r in tablo.iterrows():
    dosya = f"{HEDEF}/{r['run']}_1M.fastq.gz"
    satir = !zcat "{dosya}" | wc -l
    print(r['run'], r['genotip'], int(satir[0]) // 4, 'okuma')

!cd "{HEDEF}" && md5sum *.fastq.gz > md5sum.txt && cat md5sum.txt
print('\nHazır: Drive > melisa-ile-biyoinformatik > rna-seq > veri_1M klasörünü Zenodo\'ya yükle.')

SRR384977 WT 1000000 okuma
SRR384978 WT 1000000 okuma
SRR384979 WT 1000000 okuma
SRR384980 KO 1000000 okuma
SRR384981 KO 1000000 okuma
SRR384982 KO 1000000 okuma
2431e0349b9c0fa11f2f093aacce8f98  SRR384977_1M.fastq.gz
3905657c2c16c3bfae2ab328cc3d5ada  SRR384978_1M.fastq.gz
3266034ed3c592e4c494145de323a44b  SRR384979_1M.fastq.gz
98f1ca706bb4c08eac9f0dcdbb778e1f  SRR384980_1M.fastq.gz
f2711755464c0f59cec0f9e47564ccb3  SRR384981_1M.fastq.gz
67269b0dcc4e3be1968ecd88091039a1  SRR384982_1M.fastq.gz

Hazır: Drive > melisa-ile-biyoinformatik > rna-seq > veri_1M klasörünü Zenodo'ya yükle.
